# Implementación de un Looped Transformer (Recurrent-Depth GPT)


Se reúne en este notebook la implementación de un **Looped Transformer** (*Transformer recurrente en profundidad*), construida directamente sobre los cimientos definidos en **`base.ipynb`**.


## 1. Arquitectura General (explicación)

### 1.1. Desglose de los bloques funcionales del Looped Transformer


1. **Herencia de los componentes base (`base.ipynb`)**:
   - **Tokenizador, Datos y Bloques Fundamentales**: Se importan íntegramente `CharTokenizer`, `char_tokenizer`, `format_math_prompt`, `MathDataset`, `collate_math_batch`, `create_math_dataloader`, `IGNORE_INDEX`, `MultiHeadAttention`, `LayerNorm`, `GELU`, `FeedForward`, `TransformerBlock`, `GPTModel`, `count_params`, `block_flops_per_token`, `gpt_flops_per_token` y las funciones de generación y evaluación de `base.ipynb`.
   - **Configuración por defecto (`LOOPED_BASE_CONFIG`, `DEFAULT_TRAIN_CONFIG`) y Matriz de Experimentos (`build_experiment_matrix`, `load_experiment_configs_from_yaml`)**: Define los hiperparámetros de recurrencia (`arch="looped"`, `n_loops=4`, `n_unique_layers=1`), los hiperparámetros de entrenamiento (`AdamW`, `lr = 1e-3`, warmup de 200 pasos + coseno, `batch_size = 64`, `steps = 10000`, `grad_clip = 1.0`, precisión mixta) y el constructor de las 9 configuraciones principales $\times$ 3 semillas = 27 *runs* (más la configuración opcional `gpt_L4_d128`), con soporte directo para leer `configs/base.yaml` y `configs/experiments.yaml`.

2. **Recurrencia en Profundidad (`LoopedGPT`) y Constructor Unificado (`build_model`)**:
   - **Herencia directa de `GPTModel`**: `LoopedGPT` es una subclase de `GPTModel` que reutiliza `self.tok_emb`, `self.pos_emb`, `self.drop_emb`, `self.trf_blocks` (expuesto también mediante `@property def shared_blocks`), `self.final_norm`, `self.out_head`, `self.embed(in_idx)` y `self.head(h)`.
   - **Bucle de profundidad (`step` y `forward`)**: Calcula los embeddings de token y posición una única vez ($h_0 = \text{embed}(\text{idx})$) y aplica el bloque compartido durante $N$ iteraciones: $h_t = \text{Block}(h_{t-1})$ para $t = 1, \dots, N$. El argumento `n_loops` permite cambiar el número de iteraciones dinámicamente en inferencia sin reentrenar.
   - **Extracción unificada de estados ocultos y atenciones**: Con `return_hidden=True` y/o `return_attn=True`, tanto `GPTModel` como `LoopedGPT` devuelven `(logits, extras)` con `extras["hidden"]` ($L+1$ o $N+1$ tensores `[h0, h1, ..., hL]`) y `extras["attn"]` ($L$ o $N$ tensores `(b, n_heads, T, T)`).

3. **Contabilidad de Cómputo y Conversión de Pesos (`GPTModel` $\leftrightarrow$ `LoopedGPT`)**:
   - **Parámetros y FLOPs por token (`effective_depth`, `flops_per_token`)**: Reutiliza `count_params` y `block_flops_per_token` de `base.ipynb` y generaliza el cálculo de profundidad efectiva ($L$ en GPT, $k \cdot N$ en Looped) y FLOPs totales por token.
   - **Conversión bidireccional `GPTModel` $\leftrightarrow$ `LoopedGPT` (`unroll_to_gpt`, `looped_from_gpt`)**: Permite desplegar un `LoopedGPT` en un `GPTModel` de $L = k \cdot N$ capas con copias apiladas del bloque (produciendo exactamente los mismos *logits*), o inicializar un `LoopedGPT` a partir de un `GPTModel` (`"first"`, `"last"`, `"middle"`, `"mean"` o `"stride"`).

4. **Entrenamiento y evaluación por generación sobre problemas matemáticos**:
   - **Entrenamiento con máscara de respuesta (`train_run`)**: Optimización con `AdamW`, *warmup* lineal + decaimiento coseno, recorte de gradiente y precisión mixta automática (`bf16`/`fp16`), calculando la pérdida únicamente sobre la respuesta y `<eos>` (`ignore_index = -100`).
   - **Evaluación por generación *greedy* por nivel (`evaluate_generation_by_level`, `evaluate_loops_sweep`)**: Agrupa los problemas por longitud de pregunta (evitando *padding* a la izquierda con embeddings posicionales absolutos), genera de forma voraz hasta `<eos>` o 6 caracteres y evalúa el barrido de iteraciones $n_{\text{loops}} = 1, \dots, 2N$ sin reentrenar.

5. **Verificación Arquitectónica, Tabla de Parámetros/FLOPs y Benchmark de Hardware**:
   - **Tests de corrección (`run_architecture_tests`)**: Verifica las 7 propiedades de equivalencia, invariancia de parámetros, causalidad, invarianza de logits, proyección con `head(hidden[-1])` y normalización de la atención.
   - **Tabla comparativa (`build_params_flops_rows`, `format_params_flops_markdown`, `format_params_flops_latex`) y Benchmark (`benchmark_single_config`, `run_hardware_benchmark`)**: Genera las tablas en Markdown y LaTeX (`booktabs`) y mide el pico de VRAM y segundos por paso de entrenamiento.

### 1.2. Qué se reutiliza de `base.ipynb` y qué se añade desde cero


Siguiendo la misma filosofía que en `base.ipynb`, este cuaderno contiene exclusivamente las clases, funciones y configuraciones definitivas necesarias para instanciar, verificar, comparar, entrenar y evaluar un **Looped Transformer**, importando todo lo posible de `base.ipynb` e implementando desde cero la recurrencia en profundidad, la conversión de pesos, el bucle de entrenamiento/evaluación y las utilidades de verificación y benchmark.

| Origen / Bloque | Qué usamos o implementamos nosotros | Relación con `base.ipynb` |
| :--- | :--- | :--- |
| **`base.ipynb` (Importado íntegramente)** | • **Todas las clases, funciones y configuraciones de `base.ipynb`**: `CharTokenizer`, `char_tokenizer`, `format_math_prompt`, `MathDataset`, `collate_math_batch`, `create_math_dataloader`, `IGNORE_INDEX`, `BASE_CONFIG`, `GPT_CONFIG_124M`, `MultiHeadAttention`, `LayerNorm`, `GELU`, `FeedForward`, `TransformerBlock`, `GPTModel`, `count_params`, `block_flops_per_token`, `gpt_flops_per_token`, `generate_text_simple`, `generate`, `softmax_with_temperature`, `text_to_token_ids`, `token_ids_to_text`, `calc_loss_batch`, `calc_loss_loader`, `evaluate_model`, `train_model_simple`. | • Se cargan directamente en memoria en la Sección 2.1 mediante `load_notebook_definitions("base.ipynb")` sin reescribir ni duplicar ninguna línea de código. |
| **Configuración y Matriz de Experimentos (Añadido desde cero, §2)** | • Diccionarios **`LOOPED_DEFAULTS`**, **`LOOPED_BASE_CONFIG`** y **`DEFAULT_TRAIN_CONFIG`**.<br>• Funciones **`build_experiment_matrix`** y **`load_experiment_configs_from_yaml`**. | • Extienden `BASE_CONFIG` de `base.ipynb` y sincronizan con `configs/base.yaml` y `configs/experiments.yaml`. |
| **Arquitectura `LoopedGPT` y Constructor (Añadido desde cero, §3)** | • Clase **`LoopedGPT(GPTModel)`** con los métodos **`step`** y **`forward(in_idx, n_loops=None, return_hidden=False, return_attn=False)`**.<br>• Función constructora **`build_model(cfg)`**. | • **`LoopedGPT` hereda directamente de `GPTModel`**, reutilizando `tok_emb`, `pos_emb`, `drop_emb`, `trf_blocks`, `final_norm`, `out_head`, `embed` y `head` de `base.ipynb`. |
| **Contabilidad y Conversión de Pesos (Añadido desde cero, §4)** | • Funciones **`effective_depth`** y **`flops_per_token`** (usando `count_params` y `block_flops_per_token` de `base.ipynb`).<br>• Funciones **`gpt_cfg_from`**, **`unroll_to_gpt`** y **`looped_from_gpt`**. | • Permiten comparar parámetros/FLOPs y convertir pesos bidireccionalmente entre `LoopedGPT` y `GPTModel`. |
| **Entrenamiento y Evaluación por Nivel (Añadido desde cero, §5)** | • Planificador **`lr_schedule`** y contexto **`autocast_ctx`**.<br>• Evaluación por nivel: **`evaluate_teacher_forcing_by_level`**, **`evaluate_generation_by_level`** y **`evaluate_loops_sweep`**.<br>• Entrenamiento completo: **`train_run`**. | • Utilizan `MathDataset`, `create_math_dataloader`, `char_tokenizer` y la interfaz común de `GPTModel` y `LoopedGPT`. |
| **Tests de Corrección, Tablas y Benchmark (Añadido desde cero, §6)** | • Suite de validación **`run_architecture_tests`** (las 7 pruebas de corrección).<br>• Generadores de tabla **`build_params_flops_rows`**, **`format_params_flops_markdown`** y **`format_params_flops_latex`**.<br>• Benchmark de VRAM y velocidad **`benchmark_single_config`** y **`run_hardware_benchmark`**. | • Verifican empíricamente la equivalencia entre `LoopedGPT` y `GPTModel` y miden el coste computacional de toda la matriz. |

### 1.3. Decisiones de diseño

Para construir el Looped Transformer desde cero reutilizando al máximo `base.ipynb` sin duplicar código, se han tomado las siguientes decisiones de diseño:

1. **Carga dinámica de `base.ipynb` sin repetir ni una sola línea (`load_notebook_definitions`, Sección 2.1)**:
   - En lugar de volver a copiar las clases `CharTokenizer`, `MathDataset`, `MultiHeadAttention`, `LayerNorm`, `GELU`, `FeedForward`, `TransformerBlock` o `GPTModel`, se define una función de 10 líneas con la librería estándar (`json` + `exec`) que lee y ejecuta todas las celdas de código de `base.ipynb` en el espacio de nombres actual. Así cualquier cambio realizado en `base.ipynb` se propaga automáticamente a `loop.ipynb`.

2. **Herencia directa `class LoopedGPT(GPTModel)` y uso de `@property def shared_blocks` (Sección 3.1)**:
   - `LoopedGPT` **hereda directamente de `GPTModel`** llamando a `super().__init__({**cfg, "n_layers": self.n_unique_layers})`. De esta forma hereda todos los submódulos (`tok_emb`, `pos_emb`, `drop_emb`, `trf_blocks`, `final_norm`, `out_head`) y también los métodos `embed(in_idx)` y `head(h)` definidos en `GPTModel`, sin reescribirlos.
   - Además, se define `@property def shared_blocks(self): return self.trf_blocks`, lo que aporta **tres ventajas clave**:
     1. Permite referirse semánticamente a los bloques recurrentes como `self.shared_blocks` dentro del bucle de profundidad.
     2. Al ser un `@property` y no un segundo atributo `nn.Module`, PyTorch **no duplica** las claves (`trf_blocks` y `shared_blocks`) dentro de `state_dict()`.
     3. Como internamente sigue existiendo `self.trf_blocks`, cualquier función de `base.ipynb` que inspeccione `model.trf_blocks` funciona directamente sobre un `LoopedGPT`.

3. **Interfaz idéntica `forward(in_idx, n_loops=None, return_hidden=False, return_attn=False)` en ambos modelos (Sección 3.1 y Sección 4.5 de `base.ipynb`)**:
   - Tanto `GPTModel` como `LoopedGPT` devuelven únicamente `logits` durante el entrenamiento rápido (`return_hidden=False, return_attn=False`), o bien la tupla `(logits, extras)` durante el análisis de representaciones y atención, donde `extras["hidden"]` contiene los $L+1$ (o $N+1$) tensores `[h0, h1, ..., hN]` y `extras["attn"]` contiene los $L$ (o $N$) tensores de atención tras el `softmax`.

4. **Agrupación por longitud de pregunta en la evaluación por generación (Sección 5.2)**:
   - Dado que los embeddings posicionales de `GPTModel` y `LoopedGPT` son absolutos (`0..T-1`), rellenar por la izquierda para evaluar en batch desplazaría las posiciones de la pregunta. En `evaluate_generation_by_level`, los problemas se agrupan automáticamente por la longitud exacta de su prefijo `"Pregunta: ...\nRespuesta: "`, permitiendo generar de forma *greedy* en batch sin introducir *padding* a la izquierda.

5. **Ahorro de memoria sin cambiar el resultado (`grad_accum`, `grad_checkpoint`, Secciones 3.1, 5.3 y 6.3)**:
   - `train_run` parte cada paso en `grad_accum` trozos de `batch_size / grad_accum` ejemplos y normaliza la pérdida por el total de tokens de respuesta del *batch* completo; el gradiente es el mismo que con un único *batch* de 64. En `configs/experiments.yaml` está a 2 en los modelos de profundidad efectiva 8 y 12, que con *batch* 64 completo no caben en una GPU de 4 GB.
   - `LoopedGPT.step` aplica los bloques con `self._run_block` (definido en `GPTModel`), de modo que el *checkpointing* de activaciones opcional (`grad_checkpoint`) también funciona en cada iteración del bucle de profundidad. Junto con la atención eficiente de `base.ipynb` (§1.2, punto 7) es lo que permite entrenar toda la matriz en 4 GB.

6. **Resultados que se guardan por run (Secciones 5.2 y 5.3)**: además de la exactitud por nivel, `evaluate_generation_by_level` calcula la exactitud total y por tipo de problema (`acc_total`, `acc_by_type`: A, B ordenado y B desordenado) y puede devolver la respuesta generada para cada problema. `train_run` evalúa también `test_tpl` (redacciones reservadas), guarda las predicciones en `results/<run>_preds.jsonl` y, en los modelos looped, hace el barrido de 1 a 2N loops tanto en `test_id` (`loops_sweep`) como en `test_ood` (`loops_sweep_ood`).


## 2. Importación de `base.ipynb`, Configuración por Defecto y Matriz de Experimentos


Cargamos en el espacio de nombres actual todas las clases, funciones y configuraciones definidas en `base.ipynb` y definimos los hiperparámetros por defecto junto con el generador de la matriz de 9 configuraciones $\times$ 3 semillas.


### 2.1. Carga de las definiciones de `base.ipynb`

In [ ]:
import json
import math
import os
import platform
import random
import re
import time
from collections import defaultdict
from contextlib import nullcontext
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import yaml
except ImportError:
    yaml = None


def load_notebook_definitions(nb_path="base.ipynb", namespace=None):
    """Busca y ejecuta todas las celdas de código de `nb_path` en el espacio de nombres indicado."""
    if namespace is None:
        namespace = globals()
    candidate_paths = [
        Path(nb_path),
        Path("notebooks") / nb_path,
        Path(__file__).resolve().parent / nb_path if "__file__" in globals() else Path(nb_path),
    ]
    resolved = next((p for p in candidate_paths if p.is_file()), None)
    if resolved is None:
        raise FileNotFoundError(f"No se encontró el cuaderno base en: {[str(p) for p in candidate_paths]}")
    with open(resolved, "r", encoding="utf-8") as f:
        nb = json.load(f)
    for cell in nb.get("cells", []):
        if cell.get("cell_type") == "code":
            source = cell.get("source", [])
            code = "".join(source) if isinstance(source, list) else source
            exec(compile(code, str(resolved), "exec"), namespace)


load_notebook_definitions("base.ipynb")


### 2.2. Configuración por defecto y Matriz de experimentos: `LOOPED_BASE_CONFIG`, `DEFAULT_TRAIN_CONFIG` y `build_experiment_matrix`

In [ ]:
LOOPED_DEFAULTS = {
    "arch": "looped",         # "gpt" | "looped"
    "n_loops": 4,             # N: número de iteraciones recurrentes del bloque
    "n_layers": 1,            # k: bloques con pesos distintos (1 en Looped estándar)
    "n_unique_layers": 1,     # alias de n_layers para claridad en bloques compartidos
    "input_injection": False, # Suma opcional de h0 antes de cada iteración t > 0
}

# Configuración base adaptada a LoopedGPT
LOOPED_BASE_CONFIG = {
    **BASE_CONFIG,
    **LOOPED_DEFAULTS,
}

# Hiperparámetros de entrenamiento por defecto
DEFAULT_TRAIN_CONFIG = {
    "lr": 1e-3,
    "betas": (0.9, 0.95),
    "weight_decay": 0.1,
    "warmup_steps": 200,
    "min_lr_ratio": 0.1,
    "batch_size": 64,         # batch efectivo; con grad_accum: N se procesa en N trozos de batch_size/N
    "grad_accum": 1,
    "grad_checkpoint": False, # True = menos memoria, ~30 % más lento, mismo resultado
    "steps": 42000,           # presupuesto validado en la calibración (configs/base.yaml)
    "eval_every": 1000,
    "grad_clip": 1.0,
    "amp": True,
    "seeds": (0, 1, 2),
}


def build_experiment_configs(include_optional=True):
    """Devuelve las configuraciones de modelos de la matriz experimental (sin expandir por semilla),
    igual que configs/experiments.yaml:
    - 8 obligatorias: GPT L = 1, 2, 4, 8 (d = 256), GPT L = 4 con d = 128 (~mismos parámetros que
      Looped N = 4) y Looped N = 2, 4, 8 con 1 bloque único (d = 256, 8 cabezas)
    - 2 opcionales: GPT L = 12 y Looped N = 12
    Los modelos de profundidad efectiva 8 y 12 usan grad_accum = 2 (batch 64 en 2 trozos de 32: mismo
    resultado, la mitad de memoria) para caber en una GPU de 4 GB."""
    specs = []
    for L in (1, 2, 4, 8, 12):
        name = f"gpt_L{L}_d256"
        specs.append({
            **BASE_CONFIG,
            "name": name, "tag": name, "arch": "gpt",
            "n_layers": L, "n_loops": 1, "emb_dim": 256, "n_heads": 8,
            "grad_accum": 2 if L >= 8 else 1, "optional": L == 12,
        })
    for N in (2, 4, 8, 12):
        name = f"looped_N{N}_d256"
        specs.append({
            **BASE_CONFIG,
            "name": name, "tag": name, "arch": "looped",
            "n_layers": 1, "n_unique_layers": 1, "n_loops": N,
            "emb_dim": 256, "n_heads": 8,
            "grad_accum": 2 if N >= 8 else 1, "optional": N == 12,
        })
    name = "gpt_L4_d128"
    specs.append({
        **BASE_CONFIG,
        "name": name, "tag": name, "arch": "gpt",
        "n_layers": 4, "n_loops": 1, "emb_dim": 128, "n_heads": 4, "grad_accum": 1, "optional": False,
    })
    return [s for s in specs if include_optional or not s["optional"]]


def build_experiment_matrix(seeds=(0, 1, 2), include_optional=False):
    """Construye la matriz completa de entrenamientos expandiendo cada configuración por semilla."""
    specs = build_experiment_configs(include_optional=include_optional)
    runs = []
    for spec in specs:
        for seed in seeds:
            cfg = {**spec, "seed": seed, "run": f"{spec['name']}_s{seed}"}
            runs.append(cfg)
    return runs


def load_experiment_configs_from_yaml(yaml_path=None, include_optional=True):
    """Carga las configuraciones desde `configs/experiments.yaml` si está disponible,
    o utiliza `build_experiment_configs` como respaldo."""
    candidates = (
        [Path(yaml_path)]
        if yaml_path is not None
        else [Path("../configs/experiments.yaml"), Path("configs/experiments.yaml")]
    )
    resolved = next((p for p in candidates if p.is_file()), None)
    if resolved is not None and yaml is not None:
        data = yaml.safe_load(resolved.read_text(encoding="utf-8"))
        defaults = {**BASE_CONFIG, **data.get("defaults", {})}
        out = []
        for entry in data.get("experiments", []):
            if entry.get("optional", False) and not include_optional:
                continue
            merged = {**defaults, **entry, "tag": entry["name"]}
            if merged.get("arch") == "looped":
                merged.setdefault("n_unique_layers", merged.get("n_layers", 1))
            out.append(merged)
        return out
    return build_experiment_configs(include_optional=include_optional)


## 3. Arquitectura `LoopedGPT` y Constructor Unificado (`build_model`)


La clase `LoopedGPT` hereda directamente de `GPTModel` (definida en la Sección 4.5 de `base.ipynb`). Al inicializar `super().__init__` con `n_layers = n_unique_layers`, hereda sin repetir código las capas de embedding (`tok_emb`, `pos_emb`, `drop_emb`), los bloques compartidos (`trf_blocks`, accesibles también mediante el alias `shared_blocks`), la normalización final (`final_norm`), el cabezal (`out_head`) y los métodos `embed(in_idx)` y `head(h)`.


### 3.1. Clase `LoopedGPT` (heredando de `GPTModel`)

In [ ]:
class LoopedGPT(GPTModel):
    """GPT con pesos compartidos en profundidad (k bloques únicos aplicados N veces).
    Hereda submódulos, `embed(in_idx)` y `head(h)` directamente de `GPTModel`."""

    def __init__(self, cfg):
        full_cfg = {**BASE_CONFIG, **LOOPED_DEFAULTS, **cfg}
        self.n_loops = int(full_cfg["n_loops"])
        # Soporta tanto 'n_unique_layers' como 'n_layers' (por defecto 1 bloque único compartido)
        if "n_unique_layers" in cfg:
            k_blocks = int(cfg["n_unique_layers"])
        elif "n_layers" in cfg:
            k_blocks = int(cfg["n_layers"])
        else:
            k_blocks = 1
        self.n_unique_layers = k_blocks
        self.n_layers = k_blocks
        self.input_injection = bool(full_cfg.get("input_injection", False))

        full_cfg["n_layers"] = k_blocks
        full_cfg["n_unique_layers"] = k_blocks
        super().__init__(full_cfg)
        self.cfg = dict(full_cfg)

    @property
    def shared_blocks(self):
        """Alias semántico sobre `self.trf_blocks` (heredado de `GPTModel`) para referirse
        a los bloques compartidos sin duplicar parámetros en `state_dict`."""
        return self.trf_blocks

    def step(self, h, e, t, return_attn=False):
        """Aplica una iteración completa de los k bloques compartidos."""
        if self.input_injection and t > 0:
            h = h + e
        attns = [] if return_attn else None
        for block in self.shared_blocks:
            if return_attn:
                h, attn = block(h, return_attn=True)
                attns.append(attn)
            else:
                h = self._run_block(block, h)  # con checkpointing opcional (GPTModel._run_block)
        return (h, attns) if return_attn else h

    def forward(self, in_idx, n_loops=None, return_hidden=False, return_attn=False):
        """Paso hacia adelante:
        - Si `return_hidden=False` y `return_attn=False`: devuelve `logits` de forma (b, T, vocab_size).
        - En caso contrario: devuelve `(logits, extras)` con:
          * `extras['hidden']`: lista de L+1 tensores (b, T, d) [h0 = embeddings, h1, ..., hL]
          * `extras['attn']`: lista de L tensores (b, n_heads, T, T) con los pesos de atención de cada paso."""
        loops = self.n_loops if n_loops is None else int(n_loops)
        e = self.embed(in_idx)
        h = e

        if not return_hidden and not return_attn:
            for t in range(loops):
                h = self.step(h, e, t, return_attn=False)
            return self.head(h)

        extras = {}
        if return_hidden:
            extras["hidden"] = [h]
        if return_attn:
            extras["attn"] = []

        for t in range(loops):
            if return_attn:
                h, step_attns = self.step(h, e, t, return_attn=True)
                extras["attn"].extend(step_attns)
            else:
                h = self.step(h, e, t, return_attn=False)
            if return_hidden:
                extras["hidden"].append(h)

        return self.head(h), extras


### 3.2. Función constructora unificada: `build_model`

In [ ]:
def build_model(cfg):
    """Construye cualquier modelo (`GPTModel` o `LoopedGPT`) a partir de un diccionario `cfg`."""
    full_cfg = {**BASE_CONFIG, **cfg}
    arch = full_cfg.get("arch", "looped" if "n_loops" in cfg and "n_layers" not in cfg else "gpt")
    if arch == "gpt":
        return GPTModel(full_cfg)
    if arch == "looped":
        return LoopedGPT(full_cfg)
    raise ValueError(f"Arquitectura desconocida en cfg['arch']: {arch}")

## 4. Contabilidad de Cómputo y Conversión de Pesos (`GPTModel` $\leftrightarrow$ `LoopedGPT`)


Un `LoopedGPT` con $k=1$ bloque único y $N$ iteraciones tiene los mismos parámetros que `GPT L=1` pero realiza los mismos FLOPs por token que `GPT L=N`. Las siguientes funciones implementan `count_params(model)`, `flops_per_token(cfg)` y la conversión exacta entre `LoopedGPT` y `GPTModel` (utilizada para verificar que `Looped N` produce idénticos *logits* que un `GPT` de $N$ capas con copias apiladas del bloque).


### 4.1. Profundidad efectiva y FLOPs por token: `effective_depth` y `flops_per_token`

Reutilizando `count_params(model)` y `block_flops_per_token(cfg)` definidos en la Sección 4.7 de `base.ipynb`, definimos `effective_depth(cfg)` ($L$ en GPT, $k \cdot N$ en Looped) y `flops_per_token(cfg)` para comparar modelos en regímenes iso-parámetros e iso-FLOPs.

In [ ]:
def effective_depth(cfg):
    """Número total de aplicaciones de bloque por paso hacia adelante (L en GPT, k * N en Looped)."""
    arch = cfg.get("arch", "looped" if "n_loops" in cfg and "n_layers" not in cfg else "gpt")
    if arch == "gpt":
        return int(cfg["n_layers"])
    k = int(cfg.get("n_unique_layers", cfg.get("n_layers", 1)))
    return k * int(cfg["n_loops"])


def flops_per_token(cfg, seq_len=None, depth=None):
    """FLOPs totales de forward por token para cualquier configuración `cfg` (`GPTModel` o `LoopedGPT`).
    Reutiliza `block_flops_per_token` definido en `base.ipynb`."""
    full_cfg = {**BASE_CONFIG, **cfg}
    d_eff = effective_depth(full_cfg) if depth is None else int(depth)
    return d_eff * block_flops_per_token(full_cfg, seq_len=seq_len) + 2 * int(full_cfg["emb_dim"]) * int(full_cfg["vocab_size"])


# Alias por compatibilidad
model_flops_per_token = flops_per_token


### 4.2. Despliegue y compresión entre `LoopedGPT` y `GPTModel`: `gpt_cfg_from`, `unroll_to_gpt` y `looped_from_gpt`

In [ ]:
def gpt_cfg_from(cfg, n_layers):
    """Extrae un diccionario de configuración válido para `GPTModel` con `n_layers` capas."""
    full_cfg = {**BASE_CONFIG, **cfg}
    keys = ("vocab_size", "context_length", "emb_dim", "n_heads", "drop_rate", "qkv_bias")
    out = {k: full_cfg[k] for k in keys}
    out["arch"] = "gpt"
    out["n_layers"] = int(n_layers)
    out["n_loops"] = 1
    return out


@torch.no_grad()
def unroll_to_gpt(looped_model, cfg=None, n_loops=None):
    """Despliega un `LoopedGPT` (con `input_injection=False`) en un `GPTModel` de `base.ipynb`
    con L = k * N capas con copias apiladas del bloque que produce exactamente los mismos logits."""
    assert not looped_model.input_injection, "unroll_to_gpt requiere input_injection=False"
    base_cfg = looped_model.cfg if cfg is None else cfg
    loops = looped_model.n_loops if n_loops is None else int(n_loops)
    k = looped_model.n_unique_layers
    L = k * loops
    gpt = GPTModel(gpt_cfg_from(base_cfg, L))
    gpt.tok_emb.load_state_dict(looped_model.tok_emb.state_dict())
    gpt.pos_emb.load_state_dict(looped_model.pos_emb.state_dict())
    gpt.final_norm.load_state_dict(looped_model.final_norm.state_dict())
    gpt.out_head.load_state_dict(looped_model.out_head.state_dict())
    for i in range(L):
        gpt.trf_blocks[i].load_state_dict(looped_model.shared_blocks[i % k].state_dict())
    return gpt


@torch.no_grad()
def looped_from_gpt(gpt_model, cfg=None, n_loops=None, init="first", mode=None):
    """Construye un `LoopedGPT` a partir de un `GPTModel` de `base.ipynb`.
    Modos de inicialización de los k bloques compartidos (`init` o `mode`):
    - `'first'`: copia los primeros k bloques de `gpt_model`.
    - `'last'`: copia los últimos k bloques de `gpt_model`.
    - `'middle'`: copia los k bloques centrales de `gpt_model`.
    - `'mean'`: promedia los bloques de `gpt_model` con el mismo índice módulo k.
    - `'stride'`: selecciona k bloques equiespaciados a lo largo de `gpt_model.trf_blocks`."""
    chosen_mode = mode if mode is not None else init
    L = len(gpt_model.trf_blocks)
    base_cfg = dict(gpt_model.cfg if cfg is None else cfg)
    base_cfg["arch"] = "looped"
    base_cfg.setdefault("n_unique_layers", 1)
    base_cfg["n_layers"] = base_cfg["n_unique_layers"]
    if n_loops is not None:
        base_cfg["n_loops"] = int(n_loops)
    else:
        base_cfg.setdefault("n_loops", L)

    looped = LoopedGPT(base_cfg)
    looped.tok_emb.load_state_dict(gpt_model.tok_emb.state_dict())
    looped.pos_emb.load_state_dict(gpt_model.pos_emb.state_dict())
    looped.final_norm.load_state_dict(gpt_model.final_norm.state_dict())
    looped.out_head.load_state_dict(gpt_model.out_head.state_dict())

    k = looped.n_unique_layers
    if chosen_mode == "first":
        for j in range(k):
            looped.shared_blocks[j].load_state_dict(gpt_model.trf_blocks[j].state_dict())
    elif chosen_mode == "last":
        for j in range(k):
            looped.shared_blocks[j].load_state_dict(gpt_model.trf_blocks[L - k + j].state_dict())
    elif chosen_mode == "middle":
        start = max(0, (L - k) // 2)
        for j in range(k):
            looped.shared_blocks[j].load_state_dict(gpt_model.trf_blocks[start + j].state_dict())
    elif chosen_mode == "stride":
        stride = max(1, L // k)
        for j in range(k):
            looped.shared_blocks[j].load_state_dict(gpt_model.trf_blocks[min(j * stride, L - 1)].state_dict())
    elif chosen_mode == "mean":
        for j in range(k):
            idxs = list(range(j, L, k))
            avg_sd = {}
            for key in gpt_model.trf_blocks[0].state_dict():
                avg_sd[key] = sum(gpt_model.trf_blocks[i].state_dict()[key].float() for i in idxs) / len(idxs)
                avg_sd[key] = avg_sd[key].to(gpt_model.trf_blocks[0].state_dict()[key].dtype)
            looped.shared_blocks[j].load_state_dict(avg_sd)
    else:
        raise ValueError(f"Modo de inicialización desconocido: {chosen_mode}")
    return looped


## 5. Entrenamiento y Evaluación por Generación sobre Problemas Matemáticos


Implementamos el bucle de entrenamiento y la evaluación por generación: planificador de tasa de aprendizaje con *warmup* + coseno, precisión mixta (`bf16`/`fp16`), evaluación por generación *greedy* hasta `<eos>` o 6 caracteres agrupada por longitud de pregunta, barrido `loops_sweep` ($n_{\text{loops}} = 1, \dots, 2N$) y guardado de `checkpoints/<run>.pt` y `results/<run>.json`.


### 5.1. Planificador de tasa de aprendizaje, precisión mixta y evaluación rápida con *teacher forcing*

In [ ]:
def set_all_seeds(seed):
    """Fija las semillas de Python, NumPy y PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def lr_schedule(step, total_steps, base_lr=1e-3, warmup_steps=200, min_lr_ratio=0.1):
    """Warmup lineal durante `warmup_steps` y decaimiento coseno hasta `min_lr_ratio * base_lr`."""
    if step < warmup_steps:
        return base_lr * (step + 1) / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return base_lr * (min_lr_ratio + (1.0 - min_lr_ratio) * cosine)


def autocast_ctx(device, amp=True):
    """Precisión mixta en GPU: bf16 si la GPU lo soporta; si no, fp16 con GradScaler."""
    device = torch.device(device)
    if not amp or device.type != "cuda":
        return nullcontext(), None
    if torch.cuda.is_bf16_supported():
        return torch.autocast("cuda", dtype=torch.bfloat16), None
    return torch.autocast("cuda", dtype=torch.float16), torch.amp.GradScaler("cuda")


@torch.no_grad()
def evaluate_teacher_forcing_by_level(model, data_loader, device, n_loops=None, max_batches=None):
    """Calcula la entropía cruzada sobre la respuesta (`ce_answer_by_level`) y la exactitud por token
    con teacher forcing en validación/test."""
    model.eval()
    ce_sum, tok_correct, tok_count = 0.0, 0, 0
    by_level = defaultdict(lambda: [0.0, 0, 0])

    for i, batch in enumerate(data_loader):
        if max_batches is not None and i >= max_batches:
            break
        x, y, levels = batch[0].to(device), batch[1].to(device), batch[2].to(device)
        logits = model(x, n_loops=n_loops)
        ce_per_tok = F.cross_entropy(
            logits.flatten(0, 1).float(), y.flatten(), ignore_index=IGNORE_INDEX, reduction="none"
        ).view_as(y)
        valid = y != IGNORE_INDEX
        preds = logits.argmax(dim=-1)
        correct = (preds == y) & valid

        ce_sum += ce_per_tok[valid].sum().item()
        tok_correct += correct.sum().item()
        tok_count += valid.sum().item()

        for lvl in levels.unique().tolist():
            row_mask = (levels == lvl).unsqueeze(1) & valid
            n_lvl = row_mask.sum().item()
            if n_lvl > 0:
                acc_lvl = by_level[str(int(lvl))]
                acc_lvl[0] += ce_per_tok[row_mask].sum().item()
                acc_lvl[1] += correct[row_mask].sum().item()
                acc_lvl[2] += n_lvl

    denom = max(1, tok_count)
    ce_by_level = {
        k: v[0] / max(1, v[2]) for k, v in sorted(by_level.items(), key=lambda item: int(item[0]))
    }
    return {
        "ce_answer": ce_sum / denom,
        "tf_acc": tok_correct / denom,
        "ce_answer_by_level": ce_by_level,
    }

### 5.2. Evaluación por generación (*exactitud exacta por nivel*) y barrido de loops: `evaluate_generation_by_level` y `evaluate_loops_sweep`

In [ ]:
_INT_RE = re.compile(r"-?\d+")


def extract_predicted_integer(text):
    """Extrae el número entero de la respuesta generada; si no hay número, devuelve None."""
    match = _INT_RE.search(text.strip())
    return int(match.group(0)) if match is not None else None


def problem_group(rec):
    """Grupo del problema para H6: 'A', 'B ordenado' o 'B desordenado' (datos sin 'type': 'A')."""
    if rec.get("type", "A") == "A":
        return "A"
    return "B desordenado" if rec.get("shuffled") else "B ordenado"


@torch.no_grad()
def evaluate_generation_by_level(model, records_or_dataset, device, tokenizer=None,
                                 n_loops=None, max_new_tokens=6, batch_size=64, return_records=False):
    """Evaluación por generación greedy:
    - Pasa al modelo el texto hasta 'Respuesta: ' incluido.
    - Agrupa los problemas por longitud exacta del prefijo para generar en batch sin relleno por la izquierda.
    - Genera de forma greedy hasta <eos> o un máximo de `max_new_tokens` (6) caracteres.
    - Devuelve `acc_by_level`, `ce_answer_by_level`, la exactitud total `acc_total` y por tipo de problema
      `acc_by_type` (A / B ordenado / B desordenado). Con `return_records=True` añade `records`: una
      entrada por problema con la respuesta generada, para guardar las predicciones."""
    model.eval()
    tok = char_tokenizer if tokenizer is None else tokenizer
    records = records_or_dataset.records if isinstance(records_or_dataset, MathDataset) else list(records_or_dataset)
    context_size = model.pos_emb.weight.shape[0]

    # Agrupar por longitud de prompt para evitar padding por la izquierda con embeddings posicionales absolutos
    by_len = defaultdict(list)
    for rec in records:
        prompt_ids = tok.encode(format_math_prompt(rec["question"]))
        by_len[len(prompt_ids)].append((prompt_ids, int(rec["answer"]), int(rec.get("level", 1)), rec))

    correct_by_lvl = defaultdict(lambda: [0, 0])
    records_out = []
    for prompt_len, group in by_len.items():
        for start in range(0, len(group), batch_size):
            chunk = group[start : start + batch_size]
            idx = torch.tensor([item[0] for item in chunk], dtype=torch.long, device=device)
            finished = torch.zeros(len(chunk), dtype=torch.bool, device=device)
            gen_tokens = [[] for _ in chunk]

            for _ in range(max_new_tokens):
                idx_cond = idx[:, -context_size:]
                logits = model(idx_cond, n_loops=n_loops)[:, -1, :]
                idx_next = torch.argmax(logits, dim=-1, keepdim=True)
                for b_i in range(len(chunk)):
                    tok_id = int(idx_next[b_i, 0].item())
                    if not finished[b_i]:
                        if tok_id == tok.eos_id:
                            finished[b_i] = True
                        else:
                            gen_tokens[b_i].append(tok_id)
                if finished.all():
                    break
                idx = torch.cat((idx, idx_next), dim=1)

            for b_i, (_, gold_ans, lvl, rec) in enumerate(chunk):
                pred_str = tok.decode(gen_tokens[b_i])
                pred_ans = extract_predicted_integer(pred_str)
                stats = correct_by_lvl[str(lvl)]
                stats[0] += int(pred_ans == gold_ans)
                stats[1] += 1
                records_out.append({
                    "id": rec.get("id"), "level": lvl, "type": problem_group(rec),
                    "template": rec.get("template"), "answer": gold_ans, "pred": pred_ans,
                    "generated": pred_str, "correct": int(pred_ans == gold_ans),
                })

    acc_by_level = {
        k: v[0] / max(1, v[1]) for k, v in sorted(correct_by_lvl.items(), key=lambda item: int(item[0]))
    }
    loader = create_math_dataloader(records, tokenizer=tok, batch_size=batch_size, shuffle=False)
    tf_stats = evaluate_teacher_forcing_by_level(model, loader, device, n_loops=n_loops)
    by_type = defaultdict(list)
    for r in records_out:
        by_type[r["type"]].append(r["correct"])
    out = {
        "acc_by_level": acc_by_level,
        "acc_total": round(sum(r["correct"] for r in records_out) / len(records_out), 4) if records_out else 0.0,
        "acc_by_type": {t: round(sum(v) / len(v), 4) for t, v in sorted(by_type.items())},
        "ce_answer_by_level": tf_stats["ce_answer_by_level"],
    }
    if return_records:
        out["records"] = records_out
    return out


@torch.no_grad()
def evaluate_loops_sweep(model, records_or_dataset, device, tokenizer=None,
                         max_loops=None, batch_size=64):
    """Barrido de loops en test para `LoopedGPT`: evalúa `n_loops = 1, ..., 2N` sin reentrenar
    y devuelve el diccionario `loops_sweep`."""
    assert isinstance(model, LoopedGPT), "evaluate_loops_sweep solo aplica a LoopedGPT"
    max_loops = 2 * model.n_loops if max_loops is None else max_loops
    sweep = {}
    for t in range(1, max_loops + 1):
        res = evaluate_generation_by_level(
            model, records_or_dataset, device, tokenizer=tokenizer, n_loops=t, batch_size=batch_size
        )
        sweep[str(t)] = res["acc_by_level"]
    return sweep

### 5.3. Entrenamiento completo y guardado de checkpoints y resultados: `train_run`

In [ ]:
def train_run(cfg, train_records, val_records, test_id_records=None, test_ood_records=None,
              device=None, checkpoints_dir="checkpoints", results_dir="results", verbose=True,
              test_tpl_records=None):
    """Entrena y evalúa un run de la matriz de principio a fin y guarda:
    - `checkpoints/<run>.pt`: `{"cfg": cfg, "model_state": model.state_dict(), "step": step}`
    - `results/<run>.json`: diccionario con configuración, métricas y curvas de entrenamiento.
    - `results/<run>_preds.jsonl`: una línea por problema evaluado con la respuesta generada.
    Ahorro de memoria opcional (no cambia el resultado): `grad_accum: N` parte cada paso en N trozos de
    batch_size/N ejemplos y `grad_checkpoint: True` recalcula las activaciones de cada bloque en el backward."""
    cfg = {**BASE_CONFIG, **DEFAULT_TRAIN_CONFIG, **cfg}
    run_name = cfg.get("run", f"{cfg.get('arch', 'looped')}_s{cfg.get('seed', 0)}")
    device = torch.device(device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu"))
    set_all_seeds(int(cfg.get("seed", 0)))

    grad_accum = int(cfg.get("grad_accum", 1))
    if grad_accum < 1 or int(cfg["batch_size"]) % grad_accum != 0:
        raise ValueError(f"batch_size={cfg['batch_size']} tiene que ser múltiplo de grad_accum={grad_accum}")
    micro_batch = int(cfg["batch_size"]) // grad_accum
    train_loader = create_math_dataloader(
        train_records, batch_size=micro_batch, context_length=cfg["context_length"], shuffle=True
    )
    val_loader = create_math_dataloader(
        val_records, batch_size=cfg["batch_size"], context_length=cfg["context_length"], shuffle=False
    )

    model = build_model(cfg).to(device)
    model.grad_checkpoint = bool(cfg.get("grad_checkpoint", False))
    n_params = count_params(model)
    flops_tok = flops_per_token(cfg)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["lr"], betas=tuple(cfg["betas"]), weight_decay=cfg["weight_decay"]
    )
    ctx, scaler = autocast_ctx(device, amp=cfg.get("amp", True))

    steps = int(cfg.get("steps", cfg.get("total_steps", 10000)))
    warmup = int(cfg["warmup_steps"])
    eval_every = int(cfg["eval_every"])
    grad_clip = float(cfg["grad_clip"])

    curve = []
    best_val_ce = float("inf")
    best_state = None
    train_iter = iter(train_loader)
    t0 = time.time()

    model.train()
    for step in range(steps):
        # Los trozos del batch (con grad_accum = 1 es un único batch de batch_size ejemplos, como antes).
        micro_batches = []
        for _ in range(grad_accum):
            try:
                micro_batches.append(next(train_iter))
            except StopIteration:
                train_iter = iter(train_loader)
                micro_batches.append(next(train_iter))
        # La pérdida es la media sobre TODOS los tokens de respuesta del batch completo: cada trozo suma
        # su pérdida y se divide entre el total de tokens válidos del batch. Así el gradiente es el mismo
        # que con un único batch de batch_size ejemplos.
        n_valid = sum(int((mb[1] != IGNORE_INDEX).sum()) for mb in micro_batches)
        lr_now = lr_schedule(step, steps, base_lr=cfg["lr"], warmup_steps=warmup, min_lr_ratio=cfg["min_lr_ratio"])
        for group in optimizer.param_groups:
            group["lr"] = lr_now

        optimizer.zero_grad(set_to_none=True)
        loss_total = 0.0
        for x, y, _ in micro_batches:
            x, y = x.to(device), y.to(device)
            with ctx:
                logits = model(x)
                loss_sum = F.cross_entropy(
                    logits.flatten(0, 1).float(), y.flatten(), ignore_index=IGNORE_INDEX, reduction="sum"
                )
                loss = loss_sum / max(1, n_valid)
            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            loss_total += loss.item()

        if scaler is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        if step % eval_every == 0 or step == steps - 1:
            with ctx:
                val_stats = evaluate_teacher_forcing_by_level(model, val_loader, device, max_batches=16)
            model.train()
            entry = {
                "step": step,
                "train_loss": float(loss_total),
                "val_ce_answer": float(val_stats["ce_answer"]),
                "val_acc": float(val_stats["tf_acc"]),
            }
            curve.append(entry)
            if val_stats["ce_answer"] < best_val_ce:
                best_val_ce = val_stats["ce_answer"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if verbose:
                print(f"[{run_name}] step {step:5d}/{steps} | train_loss {entry['train_loss']:.4f} "
                      f"| val_ce {entry['val_ce_answer']:.4f} | val_acc {entry['val_acc']:.3f}")

    train_time_s = time.time() - t0
    last_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)

    eval_out = {}
    all_preds = []
    for split_name, split_records in (("test_id", test_id_records), ("test_tpl", test_tpl_records),
                                      ("test_ood", test_ood_records)):
        if split_records is None:
            continue
        res = evaluate_generation_by_level(model, split_records, device, return_records=True)
        for r in res.pop("records"):
            all_preds.append({"split": split_name, **r})
        eval_out[split_name] = res
    # Barrido de loops de 1 a 2N (¿ayudan más loops de los entrenados?) en test_id y en test_ood
    if isinstance(model, LoopedGPT):
        if test_id_records is not None:
            eval_out["loops_sweep"] = evaluate_loops_sweep(model, test_id_records, device)
        if test_ood_records is not None:
            eval_out["loops_sweep_ood"] = evaluate_loops_sweep(model, test_ood_records, device)

    serializable_cfg = {k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}
    if checkpoints_dir:
        os.makedirs(checkpoints_dir, exist_ok=True)
        torch.save(
            {"cfg": serializable_cfg, "model_state": model.state_dict(), "step": steps},
            os.path.join(checkpoints_dir, f"{run_name}.pt"),
        )
        torch.save(
            {"cfg": serializable_cfg, "model_state": last_state, "step": steps},
            os.path.join(checkpoints_dir, f"{run_name}_last.pt"),
        )

    record = {
        "run": run_name,
        "cfg": serializable_cfg,
        "params": n_params,
        "flops_per_token": flops_tok,
        "train_time_s": train_time_s,
        "curve": curve,
        "eval": eval_out,
    }
    if results_dir:
        os.makedirs(results_dir, exist_ok=True)
        with open(os.path.join(results_dir, f"{run_name}.json"), "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        if all_preds:
            with open(os.path.join(results_dir, f"{run_name}_preds.jsonl"), "w", encoding="utf-8", newline="\n") as f:
                for p in all_preds:
                    f.write(json.dumps(p, ensure_ascii=False) + "\n")

    return record, model

## 6. Verificación de Corrección Arquitectónica, Tabla de Parámetros/FLOPs y Benchmark de Hardware

Reunimos en esta sección las funciones que validan matemáticamente la implementación de `GPTModel` y `LoopedGPT`, generan las tablas comparativas de parámetros y coste computacional (en Markdown y LaTeX) y miden el consumo de memoria VRAM y el tiempo por paso de entrenamiento en la máquina actual.

### 6.1. Batería de pruebas de corrección arquitectónica: `run_architecture_tests`

Comprueba automáticamente las 7 propiedades estructurales de `GPTModel` y `LoopedGPT`:
1. `Looped N = 1` produce exactamente los mismos *logits* que `GPT L = 1` con los mismos pesos.
2. `Looped N` produce exactamente los mismos *logits* y estados ocultos que un `GPT` de $N$ capas con copias apiladas del bloque (`unroll_to_gpt`) para $N \in \{2, 4, 8, 12\}$.
3. El número de parámetros entrenables de `LoopedGPT` no varía con $N$.
4. **Causalidad**: modificar tokens futuros no altera los *logits* de las posiciones anteriores.
5. Activar `return_hidden=True` o `return_attn=True` no modifica los *logits*.
6. `model.head(hidden[-1])` coincide exactamente con los *logits* finales.
7. Cada fila de la matriz de atención suma $1$ y vale $0$ por encima de la diagonal principal.

In [ ]:
@torch.no_grad()
def run_architecture_tests(verbose=True):
    """Ejecuta las 7 pruebas de corrección sobre `GPTModel` y `LoopedGPT`."""
    def _small_cfg(arch="looped", n_layers=1, n_loops=4):
        return {
            **BASE_CONFIG,
            "arch": arch,
            "context_length": 64,
            "emb_dim": 64,
            "n_heads": 4,
            "n_layers": n_layers,
            "n_unique_layers": 1,
            "n_loops": n_loops,
            "drop_rate": 0.0,
            "qkv_bias": False,
        }

    # 1. Looped N=1 == GPT L=1
    torch.manual_seed(42)
    m_loop1 = build_model(_small_cfg("looped", n_layers=1, n_loops=1)).eval()
    m_gpt1 = build_model(_small_cfg("gpt", n_layers=1, n_loops=1)).eval()
    m_gpt1.load_state_dict(m_loop1.state_dict())
    idx = torch.randint(0, BASE_CONFIG["vocab_size"], (2, 16))
    assert torch.allclose(m_loop1(idx), m_gpt1(idx), atol=1e-6), "Fallo Test 1: Looped N=1 != GPT L=1"

    # 2. Looped N == unroll_to_gpt para N = 2, 4, 8, 12
    for n_loops in (2, 4, 8, 12):
        torch.manual_seed(100 + n_loops)
        m_loop = build_model(_small_cfg("looped", n_layers=1, n_loops=n_loops)).eval()
        m_unrolled = unroll_to_gpt(m_loop).eval()
        logits_l, ext_l = m_loop(idx, return_hidden=True)
        logits_g, ext_g = m_unrolled(idx, return_hidden=True)
        assert torch.allclose(logits_l, logits_g, atol=1e-5), f"Fallo Test 2 en N={n_loops}"
        assert len(ext_l["hidden"]) == n_loops + 1 == len(ext_g["hidden"])

    # 3. Invariancia de parámetros con N
    param_counts = {count_params(build_model(_small_cfg("looped", n_loops=n))) for n in (1, 2, 4, 8, 12)}
    assert len(param_counts) == 1, "Fallo Test 3: los parámetros de Looped cambian con N"

    # 4, 5, 6, 7 para ambas arquitecturas
    for arch in ("gpt", "looped"):
        cfg = _small_cfg(arch=arch, n_layers=3 if arch == "gpt" else 1, n_loops=4)
        model = build_model(cfg).eval()
        idx_a = torch.randint(0, BASE_CONFIG["vocab_size"], (2, 16))
        idx_b = idx_a.clone()
        cutoff = 8
        idx_b[:, cutoff:] = (idx_b[:, cutoff:] + 13) % BASE_CONFIG["vocab_size"]

        # 4. Causalidad
        assert torch.allclose(model(idx_a)[:, :cutoff, :], model(idx_b)[:, :cutoff, :], atol=1e-6), f"Fallo Test 4 ({arch})"

        # 5. Invariancia al pedir return_hidden / return_attn
        logits_plain = model(idx_a)
        logits_both, extras = model(idx_a, return_hidden=True, return_attn=True)
        assert torch.allclose(logits_plain, logits_both, atol=1e-6), f"Fallo Test 5 ({arch})"

        # 6. model.head(hidden[-1]) == logits
        assert torch.allclose(logits_plain, model.head(extras["hidden"][-1]), atol=1e-6), f"Fallo Test 6 ({arch})"

        # 7. Atención normalizada y causal
        T = idx_a.shape[1]
        upper_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
        for attn in extras["attn"]:
            row_sums = attn.sum(dim=-1)
            assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5), f"Fallo Test 7 suma ({arch})"
            above_diag = attn[:, :, upper_mask]
            assert torch.allclose(above_diag, torch.zeros_like(above_diag), atol=1e-7), f"Fallo Test 7 máscara ({arch})"

    if verbose:
        print("Todas las pruebas de corrección arquitectónica (7/7) superadas con éxito.")
    return True


### 6.2. Generación de la tabla de parámetros y FLOPs: `build_params_flops_rows`, `format_params_flops_markdown` y `format_params_flops_latex`

Construye las filas de parámetros y FLOPs por token para todas las configuraciones de la matriz experimental y las exporta en formato Markdown y LaTeX (`booktabs`).

In [ ]:
def build_params_flops_rows(configs=None, include_optional=True):
    """Calcula parámetros, bloques distintos, profundidad efectiva y FLOPs/token para cada configuración."""
    if configs is None:
        configs = load_experiment_configs_from_yaml(include_optional=include_optional)
    rows = []
    for cfg in configs:
        model = build_model(cfg)
        params = count_params(model)
        flops = flops_per_token(cfg)
        rows.append({
            "name": cfg.get("name", cfg.get("tag", "model")),
            "arch": "GPT" if cfg.get("arch") == "gpt" else "Looped",
            "emb_dim": int(cfg["emb_dim"]),
            "n_heads": int(cfg["n_heads"]),
            "n_layers": int(cfg.get("n_unique_layers", cfg.get("n_layers", 1))) if cfg.get("arch") == "looped" else int(cfg["n_layers"]),
            "n_loops": 1 if cfg.get("arch") == "gpt" else int(cfg["n_loops"]),
            "effective_depth": effective_depth(cfg),
            "params": params,
            "params_m": params / 1e6,
            "flops_per_token": flops,
            "flops_m": flops / 1e6,
            "optional": bool(cfg.get("optional", False)),
        })
    return rows


def format_params_flops_markdown(rows):
    """Formatea la tabla de parámetros y FLOPs en Markdown."""
    lines = [
        "| Modelo | Arq. | $d$ | Cabezas | Bloques ($L$) | Loops ($N$) | Prof. efectiva | Parámetros | Parámetros (M) | FLOPs / token (M) |",
        "| :--- | :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |",
    ]
    for r in rows:
        tag = " *(opc.)*" if r["optional"] else ""
        lines.append(
            f"| `{r['name']}`{tag} | {r['arch']} | {r['emb_dim']} | {r['n_heads']} | "
            f"{r['n_layers']} | {r['n_loops']} | {r['effective_depth']} | "
            f"{r['params']:,} | {r['params_m']:.3f} | {r['flops_m']:.2f} |"
        )
    return "\n".join(lines)


def format_params_flops_latex(rows):
    """Formatea la tabla de parámetros y FLOPs en LaTeX (booktabs)."""
    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{llrrrrrr}",
        r"\toprule",
        r"Modelo & Arq. & $d$ & Bloques ($L$) & Loops ($N$) & Prof. & Par\'ametros (M) & FLOPs/tok (M) \\",
        r"\midrule",
    ]
    for r in rows:
        name_tex = r["name"].replace("_", r"\_")
        lines.append(
            f"\\texttt{{{name_tex}}} & {r['arch']} & {r['emb_dim']} & "
            f"{r['n_layers']} & {r['n_loops']} & {r['effective_depth']} & "
            f"{r['params_m']:.2f} & {r['flops_m']:.2f} \\\\"
        )
    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\caption{Par\'ametros entrenables y coste computacional (FLOPs por token en el pase \textit{forward} con $T=512$) para las configuraciones de la matriz experimental.}",
        r"\label{tab:params_flops}",
        r"\end{table}",
    ])
    return "\n".join(lines)


### 6.3. Benchmark de memoria VRAM y velocidad por paso: `benchmark_single_config`, `run_hardware_benchmark` y `format_benchmark_markdown`

Ejecuta pasos de entrenamiento con tensores sintéticos (`batch_size = 64`, `seq_len = 411`, el ejemplo más largo del dataset, y `grad_accum` / `grad_checkpoint` leídos de cada configuración, precisión mixta `bf16`/`fp16`, `forward + backward + clip_grad + AdamW.step()`) para medir el pico de VRAM y los segundos por paso de cada configuración y estimar la duración total de la matriz experimental.

In [ ]:
def benchmark_single_config(cfg, device=None, batch_size=64, seq_len=411,
                            warmup_steps=1, timed_steps=3, total_train_steps=42000,
                            grad_accum=None, grad_checkpoint=None):
    """Mide el pico de VRAM y el tiempo por paso de entrenamiento para una configuración `cfg`.
    `seq_len = 411` es el ejemplo más largo del dataset (el peor caso real). `grad_accum` y `grad_checkpoint`
    se leen de `cfg` si no se indican (igual que train_run)."""
    device = torch.device(device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu"))
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)

    grad_accum = int(cfg.get("grad_accum", 1) if grad_accum is None else grad_accum)
    grad_checkpoint = bool(cfg.get("grad_checkpoint", False) if grad_checkpoint is None else grad_checkpoint)
    model = build_model(cfg).to(device)
    model.grad_checkpoint = grad_checkpoint
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95), weight_decay=0.1)
    ctx, scaler = autocast_ctx(device, amp=True)

    vocab_size = int(cfg.get("vocab_size", BASE_CONFIG["vocab_size"]))
    micro = batch_size // grad_accum  # igual que train_run: grad_accum trozos de batch_size/grad_accum
    x = torch.randint(0, vocab_size, (micro, seq_len), device=device)
    y = torch.randint(0, vocab_size, (micro, seq_len), device=device)

    def _one_step():
        optimizer.zero_grad(set_to_none=True)
        for _ in range(grad_accum):
            with ctx:
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1)) / grad_accum
            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()
        if scaler is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    for _ in range(warmup_steps):
        _one_step()
    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)

    t0 = time.perf_counter()
    for _ in range(timed_steps):
        _one_step()
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - t0

    sec_per_step = elapsed / max(1, timed_steps)
    peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if device.type == "cuda" else 0.0
    est_hours_run = (sec_per_step * total_train_steps) / 3600.0

    del model, optimizer, x, y
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "name": cfg.get("name", cfg.get("tag", "model")),
        "arch": cfg.get("arch", "looped"),
        "effective_depth": effective_depth(cfg),
        "batch_size": batch_size,
        "seq_len": seq_len,
        "grad_accum": grad_accum,
        "grad_checkpoint": grad_checkpoint,
        "sec_per_step": round(sec_per_step, 4),
        "tokens_per_sec": round((batch_size * seq_len) / sec_per_step, 1),
        "peak_vram_mb": round(peak_vram_mb, 1),
        "est_hours_per_run": round(est_hours_run, 2),
        "est_hours_3_seeds": round(est_hours_run * 3, 2),
    }


def run_hardware_benchmark(configs=None, device=None, batch_size=64, seq_len=411,
                           warmup_steps=1, timed_steps=3, total_train_steps=42000):
    """Ejecuta `benchmark_single_config` sobre todas las configuraciones indicadas."""
    if configs is None:
        configs = load_experiment_configs_from_yaml(include_optional=True)
    return [
        benchmark_single_config(
            cfg, device=device, batch_size=batch_size, seq_len=seq_len,
            warmup_steps=warmup_steps, timed_steps=timed_steps, total_train_steps=total_train_steps,
        )
        for cfg in configs
    ]


def format_benchmark_markdown(bench_rows, device_desc="Auto", total_steps=42000):
    """Formatea los resultados del benchmark de hardware en una tabla Markdown."""
    lines = [
        f"**Dispositivo medido**: {device_desc}  ",
        f"**Configuración de paso**: `batch_size = {bench_rows[0]['batch_size']}`, `seq_len = {bench_rows[0]['seq_len']}`",
        "",
        f"| Modelo | Prof. | Memoria | Segundos / paso | Tokens / s | VRAM Pico (MB) | Horas est. (1 run, {total_steps} pasos) | Horas est. (3 semillas) |",
        "| :--- | ---: | :--- | ---: | ---: | ---: | ---: | ---: |",
    ]
    total_1_seed = sum(r["est_hours_per_run"] for r in bench_rows)
    total_3_seeds = sum(r["est_hours_3_seeds"] for r in bench_rows)
    for r in bench_rows:
        vram_str = f"{r['peak_vram_mb']:.1f}" if r["peak_vram_mb"] > 0 else "CPU"
        mem = f"grad_accum={r.get('grad_accum', 1)}" + (" + ckpt" if r.get("grad_checkpoint") else "")
        lines.append(
            f"| `{r['name']}` | {r['effective_depth']} | {mem} | {r['sec_per_step']:.4f} s | "
            f"{r['tokens_per_sec']:,.0f} | {vram_str} | "
            f"{r['est_hours_per_run']:.2f} h | {r['est_hours_3_seeds']:.2f} h |"
        )
    lines.append(
        f"| **TOTAL MATRIZ** | - | - | - | - | - | **{total_1_seed:.2f} h** | **{total_3_seeds:.2f} h** |"
    )
    return "\n".join(lines)
